
# Learning Objectives

We will learn about the following in this lab:

* Point-to-point and collective MPI communication routines.
* Managing the two-level hierarchy created by global and local rank of a process and how it accesses GPU(s).
* OpenMPI process mappings and its effect on application performance.

**Note:** Execution results can vary significantly based on the MPI installation, supporting libraries, workload manager, and underlying CPU and GPU hardware configuration and topology. The codes in this lab have been tested on NERSC Perlmutter with Cray MPI 9.1, gcc 14.3 and cuda 13.2, as well as on DGX-1 8 Tesla V100 16 GB nodes connected by Mellanox InfiniBand NICs running OpenMPI v4.1.1 with HPCX 2.8.1 and CUDA v11.3.0.0 and DGX with 8 Ampere A100 80GB nodes (OpenMPI v4.1.1, HPC SDK 22.7, HPCX 2.11).

## MPI Inter-Process Communication

Let us learn more about how MPI communicates between processes.

### Point-to-Point communication

Two MPI processes can communicate directly (point-to-point) by sending and receiving data packets to and from each other. Both the sender and receivers processes must acknowledge the transaction using `MPI_Send` and `MPI_Recv` functions. MPI allows tagging messages to differentiate between various messages that processes may send to each other.

The function syntax for `MPI_Send` is:

```c
int MPI_Send(void* data, int count, MPI_Datatype datatype, int destination, 
         int tag, MPI_Comm communicator);
```

Similarly, the syntax for `MPI_Recv` is:

```c
int MPI_Recv(void* data, int count, MPI_Datatype datatype, int source, int tag,
         MPI_Comm communicator, MPI_Status* status);
```
   
A simple 2-process send-receive code is as follows:

```c
int data;
if (rank == 0) {
    data = -1;
    MPI_Send(&data, 1, MPI_INT, 1, 0, MPI_COMM_WORLD);
} else if (rank == 1) {
    MPI_Recv(&data, 1, MPI_INT, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
}
```

There are several other functions to send and receive data synchronously and asynchronously. In particular, we will make use of `MPI_SendRecv` function which sends and receives a message, and whose syntax is as follows:

```c
int MPI_Sendrecv(const void *sendbuf, int sendcount, MPI_Datatype sendtype,
                int dest, int sendtag,
                void *recvbuf, int recvcount, MPI_Datatype recvtype,
                int source, int recvtag,
                MPI_Comm comm, MPI_Status *status);
```

### Collective communication

Collective communication involves participation of all processes in a communicator. It implies an implicit synchronization point among processes. Depending on the requirement, we can peform broadcast, scatter, gather, reduce, and other operations between the participating processes. 

In our application, we would like to reduce all the rank-local norms to a single global norm using the sum operation. We use the `MPI_Allreduce` function for it which combines and reduces values from all processes and distributes the result back to all processes, and whose syntax is as follows:

```c
int MPI_Allreduce(const void *sendbuf, void *recvbuf, int count,
                  MPI_Datatype datatype, MPI_Op op, MPI_Comm comm);
```

The `op` in our case will be `MPI_SUM`.

## Communication Models

We will use multiple ranks within our program as we will use multiple nodes. There are three major approaches to handle GPUs within a node:

1. Single GPU per rank
  * One process controls one GPU.
  * Easier to program and understand.
  * We can re-use our domain decomposition approach.


2. Multiple GPUs per rank
  * Usually, all GPUs within a node are handled by one process.
  * Coordinating between GPUs is quite tricky as CUDA-based communication is intertwined with MPI communication.
  * Requires a new decomposition for the two-tier communication hierarchy (MPI and CUDA).


3. Single GPU per multiple ranks
  * Multiple processes use the same GPU and number of processes in a node is usually equal to number of cores.
  * Intended for heterogeneous codes where both CPU and GPU accelerate the application.
  * CUDA Multi-Process-Service (MPS) is required to allow multiple CUDA processes to share a single GPU context.
  
We will take the first approach due to its simplicity (which eliminates approach #2) and because our application doesn't utilize CPU for compute (which eliminates approach #3). Thus our rank (core) to GPU mapping is one-to-one, as follows:

![mpi_overview](../../images/mpi_overview.png)

### Nodel-Level Local Rank

As we will run on multiple nodes, for example 2 nodes, the number of processes launched will be 16 ( assuming 8 GPU per node like in DGX). This requires addional mapping of process id to GPU Device ID, which runs from 0 to 7 on each node. Thus, we need to create a local rank at the node level.

To achieve this, we split the `MPI_COMM_WORLD` communicator between the nodes and store it in a `local_comm` communicator. Then, we get the local rank by calling the familiar `MPI_Comm_rank` function. Finally, we free the `local_comm` communicator as we don't require it anymore. 

The code snippet to obtain the `local_rank` at each node level is as follows:

```c
int local_rank = -1;
MPI_Comm local_comm;
MPI_Comm_split_type(MPI_COMM_WORLD, MPI_COMM_TYPE_SHARED, rank, MPI_INFO_NULL, &local_comm);
MPI_Comm_rank(local_comm, &local_rank);
MPI_Comm_free(&local_comm);
```

## Implementation Exercise: Part 1

### Code Structure

Open the [jacobi_memcpy_mpi.cpp](../../source_code/mpi/jacobi_memcpy_mpi.cpp) file and the [jacobi_kernels.cu](../../source_code/mpi/jacobi_kernels.cu) files. Alternatively, you can navigate to `CFD/English/C/source_code/mpi/` directory in Jupyter's file browser in the left pane. Then, click to open the `jacobi_memcpy_mpi.cpp` and `jacobi_kernels.cu` files.

We separate the device kernels from other CUDA and MPI functions as the `nvc++` compiler is required to compile CUDA C++ which may not be installed on some platforms Note that NVIDIA's HPC SDK includes the `nvc++` compiler.

Review the [Makefile](../../source_code/mpi/Makefile) to see that we compile the CUDA kernels using `nvcc` and link the object file with `jacobi_memcpy_mpi.cpp` using the `CC` compiler as follows:

```bash
# Compiling jacobi_kernels.cu
nvcc -gencode arch=compute_80,code=sm_80 -std=c++14 jacobi_kernels.cu -c
# Compiling and linking with jacobi_cuda_aware_mpi.cpp
CC -fopenmp -std=c++14 jacobi_cuda_aware_mpi.cpp jacobi_kernels.o -o jacobi_cuda_aware_mpi
```

The device kernels are same as in previous labs. Open `jacobi_memcpy_mpi.cpp` file and understand the flow of the program. In particular, observe the following:

1. `local_rank` is used to set the current GPU device.
2. Device kernel calls have been replaced with function wrappers for ease of compilation.
3. Rank 0 is used to calculate efficiency and other metrics, even though all ranks compute `single_gpu` function to verify multi-GPU implementation's correctness.
4. In the first set of halo exchanges, `top_halo_buf` stores the top halo copied from the device on the host which is then sent to top neighbour. Whereas `bot_halo_buf` stores the updated bottom halo received from bottom neighbour that is then copied to the device from the host.
5. In the second set of halo exchanges, `top_halo_buf` stores the updated top halo received from the top neighbour that is then copied to the device from the host. Whereas `bot_halo_buf` stores the bottom halo copied from the device to the host that is then sent to the bottom neighbour.
6. Each halo exchange is wrapped in NVTX "Halo exchange Memcpy+MPI" for ease of viewing in profiler.

### To-Do

Now, implement the following marked as `TODO: Part 1-`:

* Obtain the node-level local rank by splitting the global communicator.
* Implement the MPI portion of first set of halo exchanges using `MPI_SendRecv` as explained above.
* Implement the Memcpy operations and MPI calls for the second set of halo exchanges. Recall why `cudaMemcpyAsync` is not the correct way of implementing this MPI program.
* Reduce the rank-local L2 Norm to a global L2 norm using `MPI_Allreduce` function.

After implementing these, compile the program:

In [ ]:
!cd ../../source_code/mpi && make clean && make jacobi_memcpy_mpi

Ensure there are no compilation errors. Now, let us validate the program. 

The default grid-size of 16384 $\times$ 16384 will fully utilize all 8 GPUs.

- **To run on half a node with 2 GPUs with $4K\times16K$ grid size, use: `srun -N 1 -n 2 --ntasks-per-node=2 --gpus-per-node=2 ./jacobi_memcpy_mpi -nx 4096 -ny 16384.**


In [ ]:
!cd ../../source_code/mpi && srun -N 1 -n 2 --ntasks-per-node=2 --gpus-per-node=2 ./jacobi_memcpy_mpi -nx 4096 -ny 16384

- **To run on a full node with 4 GPUs with $8K\times16K$ grid size, use: `srun -N 1 -n 4 --ntasks-per-node=4 --gpus-per-node=4 ./jacobi_memcpy_mpi -nx 8192 -ny 16384.**
- **To run across 2 nodes with 8 GPUs with $16K\times16K$ grid size, use: `srun -N 2 -n 4 --ntasks-per-node=4 --gpus-per-node=4 ./jacobi_memcpy_mpi -nx 16384 -ny 16384.**



### Perlmutter system with 4 A100s per node
Partial results obtained from Perlmutter:

Using 2 GPUs (4K $\times$ 16K grid size)

```bash
Num GPUs: 2.
4096x16384: 1 GPU:   0.8707 s, 2 GPUs:   0.5382 s, speedup:     1.62, efficiency:    80.90
```

Using 4 GPUs, 1 node (8K $\times$ 16K grid size)
```bash
Num GPUs: 4.
8192x16384: 1 GPU:   1.7198 s, 4 GPUs:   0.5863 s, speedup:     2.93, efficiency:    73.33
```

Using 8 GPUs, 2 nodes (16K $\times$ 16K grid size)
```bash
Num GPUs: 8.
16384x16384: 1 GPU:   3.4113 s, 8 GPUs:   0.5959 s, speedup:     5.72, efficiency:    71.55
```

### DGX system with 8 Ampere A100
Partial results obtained from a DGX system with 8 A100s:

Using 2 GPUs (2K $\times$ 4K grid size)
```bash
Num GPUs: 2.
2048x4096: 1 GPU:   0.1366 s, 2 GPUs:   0.1258 s, speedup:     1.09, efficiency:    54.28 
```

Using 4 GPUs, half a node (4K $\times$ 8K grid size)
```bash
Num GPUs: 4.
4096x8192: 1 GPU:   0.4438 s, 4 GPUs:   0.1889 s, speedup:     2.35, efficiency:    58.74 
```

Using 8 GPUs, full node (16384 $\times$ 16384 grid size)
```bash
Num GPUs: 8.
16384x16384: 1 GPU:   3.3022 s, 8 GPUs:   0.6327 s, speedup:     5.22, efficiency:    65.24 
```

Using 2 nodes connected by InfiniBand (IB) NICs (16384 $\times$ 32768 grid size)
```bash
Num GPUs: 16.
16384x32768: 1 GPU:   6.5526 s, 16 GPUs:   0.6500 s, speedup:    10.08, efficiency:    63.01 
```

### DGX system with 8 Tesla V100
Partial results obtained from a DGX system with 8 Tesla V100s:

Using 2 nodes connected by InfiniBand (IB) NICs (16384 $\times$ 32768 grid size):

```bash
Num GPUs: 16.
16384x32768: 1 GPU:   8.9057 s, 16 GPUs:   0.7695 s, speedup:    11.57, efficiency:    72.34 
```

Using 4 nodes connected by InfiniBand (IB) NICs (16K $\times$ 64K grid size, 4 $\times$ the single-node's grid size):
```bash
Num GPUs: 32.
16384x65536: 1 GPU:  17.6316 s, 32 GPUs:   0.8526 s, speedup:    20.68, efficiency:    64.62
```

As the communication overhead increases due to more inter-node communication, the speed-up obtained and thus the efficiency of the application decreases. Nonetheless, our program can scale across mutliple nodes.

### Profiling

We can profile an MPI program, with data from each rank written into a separate `.nsys-rep` file:

`srun [mpi options] nsys profile [nsys options] <program>`

Let's profile the application using `nsys` (on 4 GPUs):

In [ ]:
!cd ../../source_code/mpi && mkdir output_nsys
!cd ../../source_code/mpi && srun -N 2 -n 4 --ntasks-per-node=2 --gpus-per-node=2 nsys profile --mpi-impl=mpich --trace=mpi,cuda,nvtx --force-overwrite true -o output_nsys/jacobi_memcpy_mpi_report_rank%q{SLURM_PROCID} ./jacobi_memcpy_mpi -nx 8192 -ny 16384

each MPI rank will write its own `.nsys-rep` file to the subdirectory `output_nsys`. `%q{SLURM_PROCID}` will be replaced by the rank so that each MPI rank will have a unique filename.

To view the profiler report, you would need to Download and save the report file by holding down <mark>Shift</mark> and <mark>Right-Clicking</mark> [Here](../../source_code/mpi/output_profiler/jacobi_memcpy_mpi_report.nsys-rep) and choosing Save Link As. Once done open the report via the GUI.

<!--You may notice that only 8 MPI processes are visible even though we launched 16 MPI processes. Nsight Systems displays the output from a single node and inter-node transactions (copy operations) are visible. This is for ease of viewing and doesn't impede our analysis.-->

Below is an example report (using 2 nodes):

![mpi_memcpy_overview](../../images/mpi_memcpy_overview.png)

Observe the following in the Timeline snapshot:

* Two sets of halo exchanges take place, each consisting of DtoH and HtoD CUDA Memcpy with an `MPI_Sendrecv` call in between for inter-process communication followed by an `MPI_Allreduce` call. 
* Each halo exchange takes about $45\mu$s in hardware and about $60\mu$s overall including the software overhead.
* The time between two Jacobi kernel iterations is about $200\mu$s.

However, if you scroll back in time, you might notice that not all halo exchanges take $60\mu$s. For example, here's a snapshot from near the beginning of the multi-GPU Jacobi iteration loop:

![mpi_memcpy_large_time](../../images/mpi_memcpy_large_time.png)

Here, the halo exchange takes about $1100\mu$s. MPI uses a lot of heuristics to fine-tune its call-stack and communication protocol to enhance performance. Therefore, we observe the behavior shown above where initially MPI calls take significant time but it improves in subsequent iterations.

**Solution:** The solution for this exercise is present in `source_code/mpi/solutions` directory: [jacobi_memcpy_mpi.cpp](../../source_code/mpi/solutions/jacobi_memcpy_mpi.cpp).

Note that our current implementation uses explicit host-staging for every halo copy operation. From our previous labs, we know that within a node, GPU-to-GPU communication can bypass host-staging and we implemented it using DtoD CUDA Memcpy with P2P enabled. Certainly, eliminating host-staging should improve performance. There are also inter-node communication optimizations that we can employ. 

We will learn more about both intra-node and inter-node GPU-centric MPI communication optimizations in the next lab where we will work with CUDA-aware MPI. Click below to move to the next lab:

# [Next: CUDA-aware MPI](../mpi/cuda_aware.ipynb)

Here's a link to the home notebook through which all other notebooks are accessible:

# [HOME](../../../start_here.ipynb)

---
## Links and Resources

* [Programming Concepts: MPI Point-to-Point Communication](https://cvw.cac.cornell.edu/mpip2p/p2pdef)
* [Programming Concepts: MPI Collective Communication](https://wgropp.cs.illinois.edu/courses/cs598-s15/lectures/lecture29.pdf)
* [Programming Concepts: NVIDIA Multi-Process Service](https://docs.nvidia.com/deploy/pdf/CUDA_Multi_Process_Service_Overview.pdf)
* [Documentation: MPI Processing Mapping, Ranking, and Binding](https://www.open-mpi.org/doc/current/man1/mpirun.1.php#sect12)
* [Code: Multi-GPU Programming Models](https://github.com/NVIDIA/multi-gpu-programming-models)
* [Code: GPU Bootcamp](https://github.com/gpuhackathons-org/gpubootcamp/)

Don't forget to check out additional [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources) and join our [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack) to share your experience and get more help from the community.

## Licensing
Copyright © 2022 OpenACC-Standard.org.  This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.
